# 第09章 寻优的旅程

> 模板原件：第一次先复制并改成本章实际组名，例如 `A1组-实验.ipynb`、`B2组-验证.ipynb`。原件保留；副本内同组链接也要同步改名。组号与复制步骤见本章README及模板目录README。

**知识主题：非线性方程与数值优化。** 本章为独立新增章，整合两本指定教材的内容，不替换第05章，也不拆入其他章节。

> 可填写模板：复制为本组文件后填写；正文、推导、扩展实验与角色报告尚待学生完成。原八章64学时，本章另列8学时；九章全部讲授共72学时。

## 先看一幕：机器人下山，为什么越走越远？

| 第一格：只看方向 | 第二格：跨过山谷 | 第三格：拿出局部地图 |
| --- | --- | --- |
| 机器人：“箭头朝下，我迈大步就行！” | 同伴：“你一步跨到了更高的山坡！” | 机器人：“先看曲率，试一步，再决定接不接受。” |

这是供学生绘制的三格漫画分镜，尚不是完成的插图。请在 `ABC共用-图片/` 放入本组图片，用自己的组号命名并引用；概念示意不能冒充真实迭代轨迹。

**先猜，再验证：** 对 φ(x)=x²，从x=1沿负梯度走，步长α=1.2会发生什么？方向正确为什么仍会失败？把“下山”换成解f(x)=0时，目标又有什么不同？

**四个核心概念：** 非线性迭代、下降方向与步长、Newton型方法、非线性最小二乘。

**先修：** Taylor展开、梯度与Hessian、线性方程组与最小二乘。相关知识对应前面已有章节；本章完整讲授自己的主线，不把本章任务拆给其他章。

## 四节安排（4×2学时）

| 节 | 内容 | 学时 |
| --- | --- | --- |
| 9.1 | 非线性方程：从夹逼到迭代 | 2 |
| 9.2 | 下降方向：往哪里走，走多远 | 2 |
| 9.3 | Newton与拟Newton：把曲率用起来 | 2 |
| 9.4 | 非线性最小二乘：从残差到参数 | 2 |

**深度安排：** 二分和最优性条件简要引入；重点实现标量Newton、Armijo回溯和一个小型GN/LM拟合实验。BFGS讲结构并作小规模对照；方程组Newton、Wolfe实现和完整收敛性证明选做，避免在8学时内堆满四个原书章节。

## 如何填写

每节填“问题—条件—推导—实验—解释”。与 [模板-A组-实验.ipynb](模板-A组-实验.ipynb) 保持节号和参数一致；复制后把链接改到本组副本。不要把起步演示当作整节已完成。

依赖：NumPy，使用课程环境；无GPU要求。下列四段是可运行起步演示，尚未执行保存输出，不是已完成的学生实验。


## 9.1 非线性方程：从夹逼到迭代

参考：《数值分析》第5版第7章7.1—7.4、7.7（书内212—227、233—236页）。

**先写预测：**【待填写】

**完整实验任务：** 以 x²−2=0 为主例，对比二分法、不动点迭代和Newton法。补充 x³−2x+2 从0出发的Newton二周期，并记录每步残差、步长、误差与停止原因。方程组只讲线性化结构，完整程序选做。


In [ ]:
import numpy as np

# 起步：Newton解 x²−2=0，只演示正初值；不是通用求根器。
x = 1.0
history = []
for k in range(8):
    residual = x*x - 2
    history.append((k, x, abs(residual), abs(x-np.sqrt(2))))
    if abs(residual) < 1e-12:
        break
    x -= residual/(2*x)
print("k, x, 残差, 与独立参考sqrt(2)的误差")
for row in history:
    print(row)
# 待补：二分法、不动点法、导数为零/非有限值/迭代耗尽的退出处理。


**本组实现与实际结果：**【待填写】

补充自己的代码单元，记录输出、独立依据和退出原因。

**风险核验：** 残差小不必意味着根误差小；步长小可能是停滞；Newton从任意初值收敛、任意改写都收敛均不成立。

## 9.2 下降方向：往哪里走，走多远

参考：《数值优化方法》第2章2.1—2.5、第3章3.1（书内8—34、38—45页）。

**先写预测：**【待填写】

**完整实验任务：** 对 φ(x,y)=(x²+20y²)/2 比较不同固定步长与回溯；画目标值和梯度范数随迭代变化的曲线。记录函数评估次数、最小步长和失败退出，不能只比较迭代次数。


In [ ]:
# 起步：回溯选择一次负梯度步；完整迭代由学生补写。
Q = np.diag([1.0, 20.0])
phi = lambda z: 0.5 * z @ Q @ z
z = np.array([1.0, 1.0])
g = Q @ z
p = -g
alpha, c1, beta = 1.0, 1e-4, 0.5
accepted = False
for tries in range(40):
    trial = phi(z + alpha*p)
    if np.isfinite(trial) and trial <= phi(z) + c1*alpha*(g@p):
        accepted = True
        break
    alpha *= beta
print("接受:", accepted, "步长:", alpha, "旧/新目标:", phi(z), phi(z+alpha*p))
# 待补：完整循环、梯度范数、评估次数和与固定步长的对比。


**本组实现与实际结果：**【待填写】

补充自己的代码单元，记录输出、独立依据和退出原因。

**风险核验：** 梯度为零不等于极小值，更不等于全局最优；目标值单调下降不自动保证到达驻点。信赖域是局部模型的步长约束，与第7章“可信AI”的统计问题不同。

## 9.3 Newton与拟Newton：把曲率用起来

参考：《数值分析》第7章7.4、7.7；《数值优化方法》第3章3.2—3.4（书内46—69页）。

**先写预测：**【待填写】

**完整实验任务：** 先在正定二次函数上比较梯度法与Newton，再构造Hessian不定的例子检查Newton方向是否下降。BFGS用一个小型库调用或可解释实现对照，报告所用线搜索、导数评估和线性求解成本。


In [ ]:
# 起步：同一正定二次函数上的Newton一步。
p_newton = np.linalg.solve(Q, -g)
print("Newton方向:", p_newton)
print("方向导数:", g @ p_newton)
print("一步后的点:", z + p_newton)
# 待补：不定Hessian反例、阻尼/修正策略和BFGS对照。


**本组实现与实际结果：**【待填写】

补充自己的代码单元，记录输出、独立依据和退出原因。

**风险核验：** 求解 ∇φ=0 可能得到鞍点或极大点；Newton二阶收敛和BFGS超线性收敛都有条件。阻尼缩步不能把原本的非下降方向自动变成下降方向。

## 9.4 非线性最小二乘：从残差到参数

参考：《数值优化方法》第5章5.1—5.3（书内119—134页）；5.4—5.6作为选读。

**先写预测：**【待填写】

**完整实验任务：** 使用合成模型 y(t)=a exp(−bt)，以预先给定的参数和可复现噪声生成数据。比较GN与自适应LM，考察好/坏初值、噪声和近秩亏Jacobian。报告参数误差、残差、梯度范数、评估次数及退出原因；将残差最小化与AI训练/反问题联系起来。


In [ ]:
# 起步：指数模型的一次GN步与固定阻尼步，不是完整LM算法。
t = np.linspace(0.0, 2.0, 21)
true_theta = np.array([2.0, 0.7])
y = true_theta[0] * np.exp(-true_theta[1]*t)  # 先用无噪声合成数据
theta = np.array([1.5, 0.4])
def residual_and_jacobian(theta):
    a, b = theta
    e = np.exp(-b*t)
    return a*e-y, np.column_stack((e, -a*t*e))
r, J = residual_and_jacobian(theta)
p_gn = np.linalg.lstsq(J, -r, rcond=None)[0]
lam = 0.1
# 增广最小二乘实现阻尼步，避免为求步显式形成JᵀJ。
aug_J = np.vstack((J, np.sqrt(lam)*np.eye(2)))
aug_r = np.concatenate((-r, np.zeros(2)))
p_lm = np.linalg.lstsq(aug_J, aug_r, rcond=None)[0]
print("初始残差范数:", np.linalg.norm(r))
for name, p in [("GN一步", p_gn), ("固定阻尼一步", p_lm)]:
    rr, _ = residual_and_jacobian(theta+p)
    print(name, "新参数:", theta+p, "新残差范数:", np.linalg.norm(rr))
# 待补：迭代、试探步的接受/拒绝、阻尼更新、停止判据、噪声与坏初值。


**本组实现与实际结果：**【待填写】

补充自己的代码单元，记录输出、独立依据和退出原因。

**风险核验：** Jᵀr=0不等于r=0；噪声下真实参数不必等于训练残差最小的参数。λ>0使线性子问题正定不代表非线性迭代必定达到全局最优。小残差也不保证参数可辨识。